# LLM Benchmark — Colab Runner

## Kullanım
1. **İlk kurulum**: Bölüm A'yı bir kere çalıştır (pip install, git clone)
2. **Her model için**: Bölüm B'deki config'i değiştir, B hücrelerini sırayla çalıştır
3. **Sonraki model**: B1'de modeli değiştir, B2'den itibaren tekrar çalıştır

vLLM kurulumu ve git clone sadece bir kere yapılır. Model değişikliğinde sadece vLLM restart + benchmark çalışır.

---
# A) İlk Kurulum (bir kere çalıştır)

## A1) Paket kurulumu ve GPU kontrol

In [ ]:
# HuggingFace login (gated modeller için — Llama vs.)
# Colab → sol panel → 🔑 Secrets → HF_TOKEN ekle
import os
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    !huggingface-cli login --token {hf_token} --add-to-git-credential
    print('✅ HuggingFace login başarılı')
except Exception as e:
    print(f'⚠️ HF_TOKEN bulunamadı (gated modeller çalışmaz): {e}')
    print('   Colab sol panel → 🔑 Secrets → HF_TOKEN ekle')

!pip -q install -U vllm pyngrok httpx pydantic hypothesis nest_asyncio
!nvidia-smi

## A2) Proje klonu ve dizinler

In [ ]:
import os, sys

ROOT = "/content/llm"
LOG_DIR = f"{ROOT}/logs"
CACHE_DIR = f"{ROOT}/cache"
PROJECT_DIR = f"{ROOT}/benchmark"

for p in [ROOT, LOG_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_DIR

REPO_URL = "https://github.com/orhan-kaplan/benchmark.git"

if os.path.exists(PROJECT_DIR):
    print("📦 Repo mevcut, güncelleniyor...")
    !git -C {PROJECT_DIR} pull --ff-only
else:
    print("📦 Repo klonlanıyor...")
    !git clone {REPO_URL} {PROJECT_DIR}

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Doğrulama
assert os.path.exists(f"{PROJECT_DIR}/benchmark/__init__.py"), "benchmark/ bulunamadı!"
assert os.path.exists(f"{PROJECT_DIR}/benchmark_data/models.json"), "benchmark_data/ bulunamadı!"
print(f"✅ Proje hazır: {PROJECT_DIR}")

---
# B) Model Test Döngüsü (her model için tekrarla)

## B1) Model seç — ID veya isim yaz

In [ ]:
import json

#  ╔══════════════════════════════════════════════════════════════╗
#  ║  SADECE BU SATIRI DEĞİŞTİR (ID veya isim)                 ║
#  ╚══════════════════════════════════════════════════════════════╝
MODEL = 1  # ID (sayı) veya isim (string): 1, "qwen3-0.6b", 11, "qwen3-32b" vs.

# JSON'dan tüm config'i oku
with open(f"{PROJECT_DIR}/benchmark_data/models.json") as f:
    models_data = json.load(f)['models']

# ID veya isim ile bul
m = None
if isinstance(MODEL, int):
    m = next((x for x in models_data if x.get('id') == MODEL), None)
elif isinstance(MODEL, str):
    m = next((x for x in models_data if x['name'] == MODEL), None)

if m is None:
    print(f"❌ Model bulunamadı: {MODEL}")
    print(f"\nMevcut modeller:")
    for x in models_data:
        print(f"  {x.get('id', '?'):2}. {x['name']}")
    raise ValueError(f"Geçersiz model: {MODEL}")

MODEL_REPO = m['repo']
MODEL_LABEL = m['name']
MAX_MODEL_LEN = m.get('vllm', {}).get('max_model_len', 4096)
GPU_MEMORY_UTILIZATION = m.get('vllm', {}).get('gpu_memory_utilization', 0.90)
EXTRA_VLLM_ARGS = m.get('vllm', {}).get('extra_args', [])
TEST_SETS = m.get('benchmark', {}).get('test_sets', ['coding'])
TEMPERATURE = m.get('benchmark', {}).get('temperature', 0.7)
MAX_TOKENS = m.get('benchmark', {}).get('max_tokens', 1024)
TOP_P = m.get('benchmark', {}).get('top_p', 1.0)
VLLM_PORT = 8090
VLLM_BASE_URL = f"http://localhost:{VLLM_PORT}"

print(f"[{m.get('id')}] {MODEL_LABEL}")
print(f"    Repo: {MODEL_REPO}")
print(f"    vLLM: max_len={MAX_MODEL_LEN}, gpu_util={GPU_MEMORY_UTILIZATION}")
print(f"    Test: {TEST_SETS}")

## B2) Önceki modeli durdur, yeni modeli başlat

In [ ]:
import subprocess, time, requests

LOG_PATH = f"{LOG_DIR}/vllm-{MODEL_LABEL}.log"

# Önceki vLLM'i durdur
print("🛑 Önceki vLLM durduruluyor...")
subprocess.run("pkill -f 'vllm serve' || true", shell=True, check=False)
time.sleep(3)

# Eski model cache'ini temizle (disk alanı için)
# Yorum satırını kaldırırsan önceki model silinir, tekrar indirmesi gerekir
# !rm -rf {CACHE_DIR}/hub/models--*

# Yeni vLLM'i başlat
cmd = [
    "vllm", "serve", MODEL_REPO,
    "--host", "0.0.0.0",
    "--port", str(VLLM_PORT),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
    "--trust-remote-code",
    *EXTRA_VLLM_ARGS,
]

with open(LOG_PATH, "w") as f:
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)

print(f"🚀 vLLM başlatıldı (PID: {proc.pid})")
print(f"   Model: {MODEL_REPO}")
print(f"   Log: {LOG_PATH}")

# Canlı durum takibi ile bekle
def get_log_tail(n=5):
    try:
        with open(LOG_PATH, 'r') as f:
            return ''.join(f.readlines()[-n:])
    except:
        return ''

def detect_phase(log):
    l = log.lower()
    if 'error' in l or 'traceback' in l: return '❌ HATA'
    if 'downloading' in l or 'fetching' in l: return '📥 İndiriliyor'
    if 'loading model' in l or 'loading weights' in l: return '🔄 GPU\'a yükleniyor'
    if 'warming up' in l or 'cuda graph' in l: return '🔥 CUDA warmup'
    if 'started server' in l or 'uvicorn running' in l: return '✅ Hazır'
    return '⏳ Başlatılıyor'

start_time = time.time()
server_ready = False

while time.time() - start_time < 600:
    elapsed = int(time.time() - start_time)
    phase = detect_phase(get_log_tail())

    try:
        if requests.get(f"{VLLM_BASE_URL}/health", timeout=3).status_code == 200:
            server_ready = True
            break
    except: pass

    if proc.poll() is not None:
        print(f"❌ vLLM çöktü! (exit: {proc.returncode})")
        print(get_log_tail(20))
        break

    print(f"[{elapsed:3d}s] {phase}")
    if '❌' in phase:
        print(get_log_tail(15))
        break
    time.sleep(5)

if server_ready:
    print(f"\n✅ Server hazır! ({int(time.time()-start_time)}s)")
    models = requests.get(f"{VLLM_BASE_URL}/v1/models", timeout=5).json()
    for m in models.get('data', []):
        print(f"   Model: {m['id']}")
    r = requests.post(f"{VLLM_BASE_URL}/v1/chat/completions",
        json={'model': MODEL_REPO, 'messages': [{'role':'user','content':'Say hello'}], 'max_tokens': 32}, timeout=60)
    if r.status_code == 200:
        print(f"   💬 {r.json()['choices'][0]['message']['content'][:150]}")
else:
    print(f"\n❌ Timeout! Son log:\n{get_log_tail(20)}")

## B3) Benchmark çalıştır

In [ ]:
import asyncio, importlib
from pathlib import Path

# Modülleri yeniden yükle (git pull sonrası değişiklikler için)
import benchmark.runner, benchmark.catalog, benchmark.test_sets
import benchmark.api_client, benchmark.metrics, benchmark.storage, benchmark.models
for mod in [benchmark.models, benchmark.storage, benchmark.metrics,
            benchmark.api_client, benchmark.catalog, benchmark.test_sets, benchmark.runner]:
    importlib.reload(mod)

from benchmark.runner import BenchmarkRunner
from benchmark.catalog import ModelCatalog
from benchmark.test_sets import TestSetManager
from benchmark.api_client import APIClient
from benchmark.metrics import MetricsCollector, VRAMTracker
from benchmark.storage import StorageManager
from benchmark.models import BenchmarkRunConfig, GenerationParams, ModelEntry, Backend

data_path = Path(PROJECT_DIR) / "benchmark_data"
storage = StorageManager(data_path)
catalog = ModelCatalog(data_path)
test_sets = TestSetManager(data_path)
api_client = APIClient(timeout=180.0, max_retries=2)
metrics_collector = MetricsCollector()
vram_tracker = VRAMTracker()

# Model endpoint güncelle
if catalog.get(MODEL_LABEL) is None:
    catalog.register(ModelEntry(name=MODEL_LABEL, repo=MODEL_REPO, format='HF',
                               backend=Backend.VLLM, tags=[], api_endpoint=VLLM_BASE_URL))
    print(f"Model eklendi: {MODEL_LABEL}")
else:
    catalog.update(MODEL_LABEL, {'api_endpoint': VLLM_BASE_URL})
    print(f"Endpoint güncellendi: {MODEL_LABEL} → {VLLM_BASE_URL}")

runner = BenchmarkRunner(catalog=catalog, test_set_manager=test_sets, api_client=api_client,
                         metrics_collector=metrics_collector, storage=storage, vram_tracker=vram_tracker)

async def run_benchmarks():
    ids = []
    for ts_name in TEST_SETS:
        print(f"\n{'='*60}")
        print(f"Benchmark: {ts_name} × {MODEL_LABEL}")
        print(f"{'='*60}")
        config = BenchmarkRunConfig(test_set_name=ts_name, model_names=[MODEL_LABEL],
            params=GenerationParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS, top_p=TOP_P))
        result = await runner.run(config)
        ids.append(result.run_id)
        ok = sum(1 for r in result.results if r.success)
        fail = sum(1 for r in result.results if not r.success)
        print(f"✅ run_id={result.run_id} | Başarılı: {ok}, Başarısız: {fail}")
    await api_client.close()
    return ids

try:
    loop = asyncio.get_running_loop()
    import nest_asyncio; nest_asyncio.apply()
    run_ids = asyncio.run(run_benchmarks())
except RuntimeError:
    run_ids = asyncio.run(run_benchmarks())

print(f"\n🏁 Tamamlandı. Run ID'ler: {run_ids}")

## B4) Sonuçlar

In [ ]:
from benchmark.reporter import ReportGenerator
reporter = ReportGenerator(storage=storage)

for run_id in run_ids:
    print(f"\n{'='*60}")
    print(f"Rapor: {run_id}")
    print(f"{'='*60}")
    summary = reporter.generate_summary(run_id)
    print(f"  Prompt: {summary.total_prompts} | Model: {summary.total_models} | ✅ {summary.successful_results} | ❌ {summary.failed_results}")
    print()
    results = storage.read_results(run_id)
    for r in results:
        s = '✅' if r.get('success') else '❌'
        m = r.get('metrics') or {}
        t = m.get('total_time_ms') or 0
        tps = m.get('tokens_per_second') or 0
        ttft = m.get('ttft_ms')
        ttft_s = f'{ttft:.0f}ms' if ttft else '-'
        tok = m.get('completion_tokens') or 0
        print(f"  {s} {r['prompt_id']:15s} | {t:7.0f}ms | {tps:6.1f} tok/s | TTFT: {ttft_s:>7s} | {tok} tok")
        if r.get('error'): print(f"     ⚠️ {r['error'][:100]}")
    reporter.export_json(run_id, storage.runs_dir / run_id / 'report.json')
    reporter.export_csv(run_id, storage.runs_dir / run_id / 'report.csv')

## B5) Yanıtları incele

In [ ]:
INSPECT_RUN = run_ids[0] if run_ids else ''
INSPECT_PROMPT = None  # None = hepsi, veya 'code-001'

if INSPECT_RUN:
    for r in storage.read_results(INSPECT_RUN):
        if INSPECT_PROMPT and r['prompt_id'] != INSPECT_PROMPT: continue
        print(f"\n{'─'*60}")
        print(f"Prompt: {r['prompt_id']} | Model: {r['model_name']}")
        print(f"{'─'*60}")
        print(r.get('response_text', r.get('error', 'Yanıt yok'))[:3000])

## B6) Sonuçları indir

In [ ]:
import shutil
from google.colab import files

for run_id in run_ids:
    run_dir = str(storage.runs_dir / run_id)
    zip_name = f"{MODEL_LABEL}_{run_id}"
    zip_path = f"/content/{zip_name}"
    shutil.make_archive(zip_path, 'zip', run_dir)
    print(f"📦 {zip_name}.zip")
    files.download(f"{zip_path}.zip")

---
# C) Opsiyonel

ngrok (dışarıdan erişim) ve keepalive.

In [ ]:
NGROK_ENABLED = False
NGROK_AUTHTOKEN = 'TOKEN'

if NGROK_ENABLED:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    ngrok.kill()
    tunnel = ngrok.connect(VLLM_PORT, 'http')
    print(f'🌐 {tunnel.public_url}')
    print(f'Open WebUI → {tunnel.public_url}/v1')
else:
    print('ngrok devre dışı')

In [ ]:
import time, requests
print('Canlı tutma aktif. Durdurmak için interrupt et.')
while True:
    try: s = '✅' if requests.get(f'{VLLM_BASE_URL}/health', timeout=5).status_code == 200 else '⚠️'
    except: s = '❌'
    print(f'{s} {time.strftime("%H:%M:%S")}')
    time.sleep(30)

---
### Yardımcı
```python
!cat /content/llm/logs/vllm-*.log | tail -50   # Log
!nvidia-smi                                      # GPU
!rm -rf /content/llm/cache/hub/models--*         # Model cache temizle
storage.list_runs()                              # Tüm run'lar
```